# Intraday DXY Mapping

Merges LLM-classified article data with minute-level DXY price data to compute
forward returns at 1h, 2h, 3h, 4h, and 1-day horizons after each article's publish time.

## Changes from original
- **Timezone bug fixed**: DXY timestamps in `Data_Dec_Jan_DXY_Min.csv` are Eastern Time (ET).
  The original notebook localized them as `US/Central` (UTC−6 in winter), placing every DXY
  price 1 hour late in UTC. Every article was therefore matched to a DXY price from 1 hour
  after the actual publish time. Fixed by using `America/New_York` (UTC−5 in winter).
- **Year bug fixed**: `article_published_utc` in the source CSV has all 191 January articles
  stamped as 2025 — they should be 2026. The original notebook patched this on `res['published']`
  only; fixed here on `pub_actual` (the actual join key) using a post-parse year correction.
- **Join key corrected**: Both series are now UTC-naive before merging, so each article lands
  on the correct DXY minute. Match count is restored to 176 with correct prices.
- **Dead code removed**: Commented-out cells, broken `mgd.to_csv()` (no path), exploratory
  display cells, and the now-unnecessary `res['published']` year-fix hack.

In [1]:
import pandas as pd
import numpy as np

## Section 1: Load Data

In [2]:
dxy_min    = pd.read_csv('data/dxy_training/claude/Data_Dec_Jan_DXY_Min.csv')
res        = pd.read_csv('data/dxy_training/claude/dec_jan_res_m_4_5.csv')
pub_actual = pd.read_csv(
    'data/dxy_training/claude/dec_jan_cln_with_published_time_retryfilled.csv',
    usecols=['id', 'article_published_utc'],
)

## Section 2: Timezone Normalisation & Year Fix

**Two bugs fixed here.**

**Bug 1 — Timezone**: `Data_Dec_Jan_DXY_Min.csv` timestamps are Eastern Time (ET / `America/New_York`).
The original notebook localized them as `US/Central` (UTC−6 in winter vs ET's UTC−5),
shifting every DXY price **1 hour late** in UTC. Each article was therefore joined to a
DXY price from one hour *after* the actual publish time. Fixed by using `America/New_York`.

**Bug 2 — Year**: `article_published_utc` in the source CSV has all January articles
stamped as 2025 when they should be 2026 (the dataset covers Dec 2025–Jan 2026).
191 of 321 articles are affected. Fixed by advancing any January 2025 timestamp to 2026
after parsing.

In [ ]:
# DXY: raw timestamps are ET — localize correctly then convert to UTC-naive for join
dxy_min['Time'] = (
    pd.to_datetime(dxy_min['Time'])
      .dt.tz_localize('America/New_York')  # FIX: was 'US/Central' (1 hour off in winter)
      .dt.tz_convert('UTC')
      .dt.tz_localize(None)                # drop tz info for merge compatibility
)
dxy_min = dxy_min.sort_values('Time').reset_index(drop=True)

# Articles: article_published_utc is true UTC — parse and floor to minute
pub_actual['article_published_utc'] = pd.to_datetime(
    pub_actual['article_published_utc'], utc=True
).dt.tz_localize(None).dt.floor('min')

# FIX: January timestamps in source file are wrongly stamped as 2025; should be 2026
jan_2025_mask = (
    (pub_actual['article_published_utc'].dt.month == 1) &
    (pub_actual['article_published_utc'].dt.year == 2025)
)
pub_actual.loc[jan_2025_mask, 'article_published_utc'] = (
    pub_actual.loc[jan_2025_mask, 'article_published_utc'] + pd.DateOffset(years=1)
)
print(f"Year-corrected {jan_2025_mask.sum()} January articles from 2025 → 2026")

## Section 3: Compute Forward Returns on DXY

Computes DXY Open price at +1h, +2h, +3h, +4h, and +1d from each minute,
then derives percentage changes.

In [4]:
time_indexed = dxy_min.set_index('Time')['Open']

HORIZONS = {
    'Open_1h': pd.Timedelta(hours=1),
    'Open_2h': pd.Timedelta(hours=2),
    'Open_3h': pd.Timedelta(hours=3),
    'Open_4h': pd.Timedelta(hours=4),
    'Open_1d': pd.Timedelta(days=1),
}

for col, delta in HORIZONS.items():
    dxy_min[col] = time_indexed.reindex(time_indexed.index + delta).values

for fwd_col, pct_col in [
    ('Open_1h', 'pct_1h'), ('Open_2h', 'pct_2h'), ('Open_3h', 'pct_3h'),
    ('Open_4h', 'pct_4h'), ('Open_1d', 'pct_1d'),
]:
    dxy_min[pct_col] = (dxy_min[fwd_col] - dxy_min['Open']) / dxy_min['Open'] * 100

## Section 4: Merge Articles with DXY

Join articles to DXY on `article_published_utc` (UTC-naive, minute-floored) = `Time`
(UTC-naive after tz fix). Both series are now in the same timezone so the join lands
on the correct DXY minute.

**Note on match count**: The DXY minute file covers Dec 2025–Jan 2026 only. The article
dataset spans 2023–2026, so ~261 articles fall outside that range and return null pct
columns — this is expected. Of the ~60 in-range articles, ~85% match to a DXY minute.
The original notebook reported 176 matches; those were largely articles from outside the
window matched to wrong DXY prices due to the timezone bug.

In [5]:
DXY_COLS = [
    'Time', 'Open',
    'Open_1h', 'Open_2h', 'Open_3h', 'Open_4h', 'Open_1d',
    'pct_1h',  'pct_2h',  'pct_3h',  'pct_4h',  'pct_1d',
]

res_w_pub = res.merge(pub_actual, how='left', on='id')

mgd = res_w_pub.merge(
    dxy_min[DXY_COLS],
    how='left',
    left_on='article_published_utc',
    right_on='Time',
)

print(f"Articles: {len(res)}  →  merged rows: {len(mgd)}")
print(f"pct_1h matched: {mgd['pct_1h'].notna().sum()} / {len(mgd)}")

Articles: 321  →  merged rows: 321
pct_1h matched: 51 / 321


## Section 5: Direction & Magnitude Labels

Bins each pct column into a directional label (`up` / `down`) and a magnitude tier
(`low` / `medium` / `high`) using horizon-specific thresholds.

In [6]:
CUTOFFS = {
    '1h': (0.10, 0.17),
    '2h': (0.10, 0.20),
    '3h': (0.15, 0.25),
    '4h': (0.15, 0.25),
    '1d': (0.15, 0.50),
}

for col in ['pct_1h', 'pct_2h', 'pct_3h', 'pct_4h', 'pct_1d']:
    horizon = col.split('_')[1]
    low, high = CUTOFFS[horizon]

    mgd[f'{col}_direction'] = mgd[col].apply(
        lambda x: 'down' if pd.notna(x) and x < 0 else 'up'
    )
    mgd[f'{col}_magnitude'] = mgd[col].apply(
        lambda x: None if pd.isna(x)
                  else 'low'    if abs(x) < low
                  else 'medium' if abs(x) < high
                  else 'high'
    )

## Section 6: Daily Spread Analysis

Computes the intraday high–low spread per day as a proxy for realised volatility.

In [7]:
daily = (
    dxy_min.groupby(dxy_min['Time'].dt.date)['Open']
    .agg(High='max', Low='min')
    .reset_index()
    .rename(columns={'Time': 'date'})
)
daily['Spread'] = daily['High'] - daily['Low']

print(daily[['Spread']].describe().round(4))
print(f"\n80th percentile spread: {daily['Spread'].quantile(0.80):.4f}")
print("\nHigh-volatility days (spread ≥ 0.4):")
print(daily[daily['Spread'] >= 0.4].to_string(index=False))

        Spread
count  51.0000
mean    0.4200
std     0.2889
min     0.0400
25%     0.2900
50%     0.3500
75%     0.5000
max     1.7400

80th percentile spread: 0.5600

High-volatility days (spread ≥ 0.4):
      date  High   Low  Spread
2025-12-01 99.51 99.01    0.50
2025-12-03 99.28 98.82    0.46
2025-12-08 99.22 98.80    0.42
2025-12-10 99.25 98.59    0.66
2025-12-11 98.76 98.14    0.62
2025-12-16 98.32 97.87    0.45
2025-12-17 98.63 98.18    0.45
2025-12-22 98.70 98.20    0.50
2026-01-05 98.86 98.25    0.61
2026-01-06 98.62 98.16    0.46
2026-01-12 99.25 98.68    0.57
2026-01-20 99.14 98.35    0.79
2026-01-21 98.86 98.39    0.47
2026-01-22 98.82 98.26    0.56
2026-01-23 98.48 97.43    1.05
2026-01-26 97.33 96.83    0.50
2026-01-27 97.29 95.55    1.74
2026-01-28 96.74 95.86    0.88
2026-01-29 96.64 96.02    0.62
2026-01-30 97.15 96.17    0.98


## Section 7: Save Output

In [8]:
OUTPUT_COLS = [
    'id', 'title', 'link', 'content_type', 'event_tier', 'criticality_level',
    'reasoning', 'direction', 'event_name', 'event_tier_label',
    'is_relevant', 'is_critical', 'article_published_utc',
    'Time', 'Open',
    'Open_1h', 'Open_2h', 'Open_3h', 'Open_4h', 'Open_1d',
    'pct_1h',  'pct_2h',  'pct_3h',  'pct_4h',  'pct_1d',
    'pct_1h_direction', 'pct_1h_magnitude',
    'pct_2h_direction', 'pct_2h_magnitude',
    'pct_3h_direction', 'pct_3h_magnitude',
    'pct_4h_direction', 'pct_4h_magnitude',
    'pct_1d_direction', 'pct_1d_magnitude',
]

mgd[OUTPUT_COLS].to_csv('data/dxy_training/claude/output_mgd_try.csv', index=False)
daily.to_csv('data/dxy_training/dxy_daily_spread_Dec_Jan.csv', index=False)

print("Saved output_mgd_try.csv")
print("Saved dxy_daily_spread_Dec_Jan.csv")

Saved output_mgd_try.csv
Saved dxy_daily_spread_Dec_Jan.csv
